[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.04.notebook-title|Demonstration: pandas -->
# Week 4 Demonstration: pandas
<!-- /gen -->

*The pandas walk-through: `Series`, `DataFrames`, file I/O, filtering, merging, and `groupby()`. It covers everything you need for [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 04, plus the habits to help you catch an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s mistakes.*

**About this notebook.** Work through it before the workshop. It takes about 75–90 minutes. As always:

1. run every cell
2. change things and re-run
3. break things and fix them

The 🤖 **[AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens)** callouts mark moments to bring in your Associate. [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) records the conversation. Set up Copilot with the [workstation guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/codespaces-start.md).

**Coverage:**

- Creating `Series` and `DataFrames`
- Selecting and filtering
- Sorting and duplicates
- Applying functions
- Reading and writing CSV, Excel, and JSON
- Missing data
- Groupby aggregation
- Merging and joining
- A <!-- plain-language: everyday first look -->first look at plotting

Nothing here needs more Python than Weeks 1-2.

## About pandas (again)

[pandas](https://pandas.pydata.org/) is the library that makes Python a data-analysis language. It is built on [NumPy](https://numpy.org/), which this notebook uses only to generate data. The [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/04/Introduction.ipynb) shows one job done once with a loop (pure Python) and once with pandas.

Further reading, both free: Wes McKinney's [*Python for Data Analysis*](https://wesmckinney.com/book/), by the author of pandas, and Corey Schafer's [pandas video series](https://www.youtube.com/watch?v=ZyhVh-qRZPA).

## Data setup

The cell below creates the files the rest of the notebook reads: A CSV file, a JSON file, and an Excel workbook. Their data is invented for a fictional client. The cell uses the module seed (`5813`), so every student and every run sees identical data.

You do not need to download anything.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# Creates the three files this notebook reads: data.csv, data.json, data.xlsx.
import os

import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week04"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

# data.csv: Six months of daily sales for a fictional homeware client
days = pd.date_range("2026-01-01", "2026-06-30", freq="D")
sales = pd.DataFrame({
    "date": days,
    "store": rng.choice(["Egham", "Staines", "Windsor"], len(days)),
    "category": rng.choice(["Lighting", "Textiles", "Decor"], len(days)),
    "units": rng.integers(5, 60, len(days)),
})
sales["revenue"] = (sales["units"] * rng.uniform(8, 25, len(days))).round(2)
sales.to_csv(f"{DATA_DIR}/data.csv", index=False)

# data.json: A small staff records file (JSON is common for web/API data)
staff = pd.DataFrame({
    "name": ["Amara", "Ben", "Chloe", "Dev", "Elena", "Farid"],
    "role": ["Analyst", "Analyst", "Senior Analyst", "Analyst",
             "Partner", "Senior Analyst"],
    "office": ["Egham", "Egham", "Windsor", "Staines", "Egham", "Windsor"],
    "salary": rng.integers(28, 75, 6) * 1000,
})
staff.to_json(f"{DATA_DIR}/data.json", orient="records")

# data.xlsx: A monthly summary workbook (needs the openpyxl library)
monthly = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "sales": rng.integers(40, 90, 6) * 1000,
    "expenses": rng.integers(25, 55, 6) * 1000,
})
with pd.ExcelWriter(f"{DATA_DIR}/data.xlsx") as writer:
    monthly.to_excel(writer, sheet_name="Sheet1", index=False)

for f in ["data.csv", "data.json", "data.xlsx"]:
    print(f"Created {DATA_DIR}/{f}")

## Setup

Import pandas and NumPy, then print their versions. Behaviour differs between major versions. **Your Associate may have learned an older one** (that's the nature of LLMs).

In [ ]:
import numpy as np
import pandas as pd

print(f"pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

## Creating and manipulating `Series`

A **`Series`** is a one-dimensional labelled array. It is ordered like a list and labelled like a dictionary. Every `DataFrame` column is a `Series`. The reference for both structures is pandas' [intro to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html).

Create a `Series` from a list. `np.nan` is pandas' marker for a missing value. Note how it displays.

In [ ]:
s = pd.Series([1, 3, 5, np.nan, 6, 8])
print(s)

The numbers on the left are the **index**. They are automatic integers unless you supply labels.

In [ ]:
s = pd.Series([1, 3, 5, 7], index=["a", "b", "c", "d"])
print(s)

Access elements **by label** with `[...]` or `.loc[...]`, and **by position** with `.iloc[...]`. Always use `.iloc` when you mean position. Your Associate may write `s[0]` to mean the first element, but modern pandas reads what is inside `[]` as a label, not a position.

In [ ]:
print(s["b"])       # by label
print(s.iloc[0])    # by position (first element)
print(s["b":"c"])   # label slicing. Note: INCLUSIVE of the end label

Operations apply to the whole `Series` at once, so you need no loop.

In [ ]:
print(s * 2)  # multiply every element

Summary statistics are built in (lucky us!). `.idxmax()` returns the *label* of the largest value (the 'maximum index number'), not the value itself. This is useful as a manager typically asks *which* row is the largest, not *how much* it is.

In [ ]:
print(s.mean())    # average
print(s.max())     # largest value
print(s.idxmax())  # the LABEL of the largest value

> 🤖 **AI Lens:** Ask your Associate the question below. Which explanation would work if you read it aloud in a client meeting?

```text
Explain how a pandas Series differs from a Python list and from a dictionary, once for a programmer, once for a manager who has only ever used Excel.
```

**A good answer** tells the programmer a `Series` is a single-typed column with an index and vectorised maths. It tells the manager it is one labelled column of a spreadsheet. The manager's version is the one to test aloud.

## Working with `DataFrames`

A **`DataFrame`** is a two-dimensional labelled table. Think of a [spreadsheet](https://youtu.be/UBX2QQHlQ_I?t=13), or a dictionary of `Series` (each column is a `Series`).

### Creating a `DataFrame`

Build one from a dictionary of columns.

In [ ]:
data = {
    "Name": ["Alice", "Bob", "Carol", "Dan"],
    "Age": [28, 34, 29, 32],
    "City": ["London", "Paris", "Berlin", "Cardiff"],
}

df = pd.DataFrame(data)
df  # the last expression in a cell displays without print(), and looks better

### Inspecting a `DataFrame`

Run `.head()`, `.info()`, and `.describe()` on every dataset you receive before you believe anything about it. The reference is pandas' [essential basic functionality guide](https://pandas.pydata.org/docs/user_guide/basics.html).

In [ ]:
print(df.head())      # first 5 rows. What does the data LOOK like?

Next, `.info()` gives the row count, each column's data type (its **`dtype`**), and how many values are missing. If you run only one check on a file you did not create, run this one.

In [ ]:
df.info()             # rows, columns, dtypes, and missing values

Then `.describe()` gives summary statistics for every numeric column. Impossible values, such as a negative age or a £0 sale, first show up here.

In [ ]:
print(df.describe())  # statistical summary of numeric columns

One more check is `.dtypes`. Text columns show the `str` `dtype`. pandas infers their type from the values. **When a column you expect to be numeric shows `str` (or `object`), something upstream went wrong.** The usual causes are a stray comma, a `£` sign, or a "TOTAL" row.

In [ ]:
print(df.dtypes)

### Selecting data

Select columns by name, rows by label (`.loc`) or position (`.iloc`), and single values with either. The [pandas indexing guide](https://pandas.pydata.org/docs/user_guide/indexing.html) covers every form.

In [ ]:
print(df["Name"])          # one column (a Series)
print()
print(df[["Name", "Age"]]) # several columns (a DataFrame; note the double brackets)

Next, select rows. `.loc` finds them by index *label*, and `.iloc` by *position*. Here the two agree because the labels are 0-3. The AI Lens below explores when they would not.

In [ ]:
print(df.loc[1])           # row with index LABEL 1
print()
print(df.iloc[1])          # row at POSITION 1 (same here, not always!)
print()
print(df.loc[1, "Name"])   # a single value: Row label 1, column "Name"

> 🤖 **AI Lens:** `.loc` and `.iloc` give the same answer above. Ask your Associate the prompt below and run what it gives you. If you can predict the difference, you understand pandas indexing.

```text
Construct a small DataFrame where `df.loc[1]` and `df.iloc[1]` return DIFFERENT rows, and explain why.
```

**A good answer** constructs a `DataFrame` whose index labels are not 0, 1, 2. It then shows the two calls returning different rows, and explains why: `.loc` looks up the label, `.iloc` counts the position.

### Adding columns

Assign to a column name that does not exist yet.

In [ ]:
df["Country"] = ["UK", "France", "Germany", "UK"]
print(df)

### Filtering rows

**Boolean indexing**: Write a condition and get back only the rows where it is `True`. This is how a business question becomes code.

In [ ]:
# Who is 30 or older?
print(df[df["Age"] >= 30])

`.isin()` answers the question *is it one of these?* It is tidier than chaining `|` conditions when the list grows.

In [ ]:
# Who is in one of these cities? .isin() checks membership in a list
print(df[df["City"].isin(["Paris", "Berlin"])])

Combine conditions with `&` (and) and `|` (or). Wrap each condition in its own parentheses. pandas requires this.

In [ ]:
# Combine conditions with & (and) / | (or). Put each condition in (parentheses)
print(df[(df["Age"] >= 30) & (df["Country"] == "UK")])

> 🤖 **AI Lens:** Ask your Associate the prompt below, with the code that built `df` pasted in place of the placeholder. Then decide which filter *you* would want to find in a colleague's notebook, and why.

```text
<paste the code that builds df here>

Show me three different ways to filter this DataFrame for people aged 30+ in the UK (boolean indexing, `.query()`, and `.loc` with a mask) and the trade-offs of each.
```

**A good answer** gives three filters that return the same rows as the cell above. Its trade-offs mention readability, `.query()` sparing you the repeated `df[...]`, and `.loc` letting you pick columns in the same call.

### Sorting

[`sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) orders rows by one column or several. `ascending=False` puts the largest first (can you guess how to put the smallest first?).

In [ ]:
print(df.sort_values("Age", ascending=False))

### Duplicates

Real exports contain accidental duplicates. You meet some in this week's [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest). Count first with [`.duplicated()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html), then decide. Never drop what you have not counted.

In [ ]:
# Add a duplicate row on purpose, then detect it
df_dup = pd.concat([df, df.iloc[[0]]], ignore_index=True)
print(f"Rows: {len(df_dup)}, duplicated: {df_dup.duplicated().sum()}")

Once you have counted and understood the duplicates, drop them. `drop_duplicates()` keeps the first copy of each row.

In [ ]:
df_clean = df_dup.drop_duplicates()
print(f"After drop_duplicates: {len(df_clean)} rows")

### Applying functions to columns

[`.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html) runs a function on every element. This is extremely powerful.

Any function works. In the second line below, the `lambda` is a one-line unnamed function (I call them 'throwaway' functions, but that's not how experts call them) that subtracts five from each age (optional background: [Week 2b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Introduction.ipynb)).

In [ ]:
df["Name_Length"] = df["Name"].apply(len)
df["Age_Minus_5"] = df["Age"].apply(lambda x: x - 5)
print(df)

`.apply()` is flexible but **slow on big data**. For arithmetic, plain column maths is faster and clearer. `df["Age"] - 5` beats the lambda above. Keep `.apply()` for logic that column maths cannot express.

> 🤖 **AI Lens:** Paste the `.apply()` code cell above in place of the placeholder and ask the question below. Associates over-use `.apply()`. This prompt teaches you to catch it.

```text
<paste the cell's code here>

Which of these `.apply()` calls could be replaced by a vectorised operation? Rewrite them, and tell me when `.apply()` is the right tool.
```

**A good answer** rewrites `df["Age"].apply(lambda x: x - 5)` as `df["Age"] - 5` and `df["Name"].apply(len)` as `df["Name"].str.len()`. It keeps `.apply()` for logic that column maths cannot express. A reply that says `.apply()` is fine as it stands has missed the point.

### Creating categories from numbers

[`pd.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) bins a numeric column into labelled **ranges**, such as age groups, price tiers, or customer segments.

In [ ]:
df["Age_Group"] = pd.cut(df["Age"],
                         bins=[0, 30, 40, 100],
                         labels=["Under 30", "30-40", "Over 40"])
print(df[["Name", "Age", "Age_Group"]])

### Quick statistics on a column

`.describe()` works on a single column too. It is the fastest <!-- plain-language: everyday first look -->first look at any numeric variable.

In [ ]:
print(df["Age"].describe())

## Reading and writing files

Now it get's interesting, because data normally arrives as **files**. The data-setup cell at the top created `data.csv`, `data.json`, and `data.xlsx`. Read them back, as you would a client's export. The [pandas I/O documentation](https://pandas.pydata.org/docs/user_guide/io.html) lists every format.

### CSV

[`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) turns a CSV file (remember what that is? Ask your Associate!) into a `DataFrame` in one line.

In [ ]:
df_sales = pd.read_csv("assets/data/week04/data.csv")
print(df_sales.head())
print(f"\n{len(df_sales)} rows")

**Check the `dtypes` immediately.** `date` came back as `str`. CSV files carry no type information, so pandas guesses every column's type on the way in. Dates need parsing.

In [ ]:
print(df_sales.dtypes)

This is a problem as we really need pandas to "understand" (I deliberaly put this in quotation marks!) dates; we find it easy, but a computer really needs to be told that this is a date. If we don't, we won't be able to sort by year, months, days, etc.

The repair is [`pd.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) with an explicit `format=` (`%d` day, `%m` month, `%Y` four-digit year). Telling pandas the format beats letting it guess. This week's quest shows why.

In [ ]:
# Parse the date column properly. Now it sorts, filters, and groups by time
df_sales["date"] = pd.to_datetime(df_sales["date"], format="%Y-%m-%d")
print(df_sales.dtypes)
print(f"\nDate range: {df_sales['date'].min():%d %B} to {df_sales['date'].max():%d %B %Y}")

`to_csv` writes a `DataFrame` back out to a file. `index=False` stops pandas writing its row numbers as an extra column.

In [ ]:
df_sales.to_csv("assets/data/week04/sales_copy.csv", index=False)
print("Written: assets/data/week04/sales_copy.csv")

### Excel

Clients often work in Excel (don't try to convince them otherwise, it's hopeless). [`read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) reads a workbook into a `DataFrame`, and `to_excel` writes one back. Both use the `openpyxl` library, which the module requirements already install.

In [ ]:
df_monthly = pd.read_excel("assets/data/week04/data.xlsx", sheet_name="Sheet1")
print(df_monthly)

Writing works the same way as CSV. Add a computed column, then write the workbook back out for the client.

In [ ]:
df_monthly["profit"] = df_monthly["sales"] - df_monthly["expenses"]
df_monthly.to_excel("assets/data/week04/monthly_with_profit.xlsx",
                    sheet_name="Sheet1", index=False)
print("Written: assets/data/week04/monthly_with_profit.xlsx")

### JSON

JSON is the common format for web and API data (remember APIs?). Its records look like Python dictionaries. [`pd.read_json`](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html) reads them.

In [ ]:
df_staff = pd.read_json("assets/data/week04/data.json")
print(df_staff)

`to_json` writes it back out. `orient="records"` writes the list-of-dictionaries shape that most web systems expect.

In [ ]:
df_staff.to_json("assets/data/week04/staff_copy.json", orient="records")
print("Written: assets/data/week04/staff_copy.json")

pandas reads other formats too, which you do not need this week. `pd.read_html` reads tables from web pages, `pd.read_sql` queries databases, and `pd.read_parquet` reads Parquet, a compressed format that large data platforms use. Each works the same way: One line reads the source into a `DataFrame`. Then you check the `dtypes`.

## Handling missing data

Real data has missing values. pandas marks them `NaN` and gives you explicit choices for handling them. The [missing-data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html) lists them. The choice is yours. It can be worth real money.

In [ ]:
df_gaps = pd.DataFrame({
    "A": [1, 2, np.nan, 4],
    "B": [5, np.nan, np.nan, 8],
    "C": [9, 10, 11, 12],
})
print(df_gaps)
print(f"\nMissing values per column:\n{df_gaps.isna().sum()}")

Option one: Drop every row that contains a missing value. This invents no values, but you lose the good values in those rows too.

In [ ]:
# Option 1: Drop any row containing a missing value
print(df_gaps.dropna())

Option two: Fill the missing values with a constant. The fill is explicit, though zero is rarely the true value.

In [ ]:
# Option 2: Fill missing values with a constant
print(df_gaps.fillna(0))

Option three: Fill with each column's mean. This is a repair. Disclose every repair to whoever reads the numbers.

In [ ]:
# Option 3: Fill with each column's mean (a simple 'repair')
print(df_gaps.fillna(df_gaps.mean(numeric_only=True)))

> 🤖 **AI Lens:** Ask your Associate the prompt below. Keep its answer in mind. Quest 04's [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) tier asks you to make exactly this call, with evidence.

```text
For a sales ledger where 4% of revenue values are missing, argue for dropping the rows, then argue for repairing them, then tell me what you would need to know to choose.
```

**A good answer** argues each case without favouring either. Then it names what decides the call: Whether the gaps are random or concentrated in one store or month, and what the report is for.

## Grouping and aggregating

**Split-apply-combine**: Split rows into groups, apply a calculation to each group, then combine the results. Every "revenue by region" question is this pattern. The [`groupby()` user guide](https://pandas.pydata.org/docs/user_guide/groupby.html) is its reference. The next cells use `df_sales`, the sales file from the data setup.

In [ ]:
# Total revenue by store, across the whole half-year
print(df_sales.groupby("store")["revenue"].sum().round(2))

`.agg()` accepts a list, so one line returns several summaries per group. The result is a table ready for a report.

In [ ]:
# Several aggregations at once with .agg()
print(df_sales.groupby("category")["revenue"].agg(["sum", "mean", "count"]).round(2))

You can group by *derived* values too. `dt.to_period("M")` turns each date into its month, so "revenue by month" is one line. This is the main step in this week's quest.

In [ ]:
# Group by MONTH using the parsed date column: The quest's main step
monthly_rev = df_sales.groupby(df_sales["date"].dt.to_period("M"))["revenue"].sum()
print(monthly_rev.round(2))

> 🤖 **AI Lens:** [TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back) practice. Write your own explanation of *split-apply-combine* in place of the placeholder in the first prompt. Ask your Associate to mark it out of 10. Then send the second prompt. Whose version is better?

```text
I am going to explain split-apply-combine in my own words. Mark my explanation out of 10, and point out anything wrong or missing.

<your explanation of split-apply-combine>
```

```text
Explain split-apply-combine to a client in three sentences, no jargon.
```

**A good answer** from you names all three steps. The steps are to split the rows into groups, apply a calculation to each group, and combine the results into one table. The Associate's client version only beats yours if it says the same in plainer words.

## Merging and joining `DataFrames`

Analysis usually means combining sources, such as a sales table and a customer file. pandas offers these tools, which are extremely powerful:

- **`pd.concat`**: Stack `DataFrames` on top of each other (same columns, more rows)
- **`pd.merge`**: Join `DataFrames` side by side on a shared key column

The [merging guide](https://pandas.pydata.org/docs/user_guide/merging.html) covers every variation. Start with two small `DataFrames` whose keys only partly overlap.

In [ ]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value_left": [1, 2, 3, 4]})
df2 = pd.DataFrame({"key": ["B", "D", "E", "F"], "value_right": [5, 6, 7, 8]})

print("df1:")
print(df1)
print("\ndf2:")
print(df2)

The `how=` argument decides **which rows the merge keeps**. pandas fills unmatched cells with `NaN`.

| `how=` | Keeps | Keys in this example | Use when |
|--------|-------|-----------|----------|
| `"inner"` | Only keys present in **both** | B, D | Matched records only (the default) |
| `"outer"` | **Every** key from either side | A, B, C, D, E, F | You cannot afford to lose a row |
| `"left"` | All of the **left** frame | A, B, C, D | Enriching a main table with lookups |
| `"right"` | All of the **right** frame | B, D, E, F | Like left, the other way round |

In [ ]:
print("Inner join (only B and D match both sides):")
print(pd.merge(df1, df2, on="key", how="inner"))

The same merge with `how="outer"` keeps every key, and the gaps show as `NaN`.

In [ ]:
print("Outer join (everyone, gaps filled with NaN):")
print(pd.merge(df1, df2, on="key", how="outer"))

With `how="left"`, every `df1` row stays, matched or not. This is the shape of most "enrich my main table" jobs.

In [ ]:
print("Left join (all of df1, matches where df2 has them):")
print(pd.merge(df1, df2, on="key", how="left"))

Pass `indicator=True` while developing a merge. It adds a column showing where each row came from, so you can see which rows found no match.

In [ ]:
check = pd.merge(df1, df2, on="key", how="outer", indicator=True)
print(check)
print(f"\nRow counts: {check['_merge'].value_counts().to_dict()}")

> 🤖 **AI Lens:** Give your Associate the scenario below. Check its answer against the `how=` table in this section. The default `how="inner"` quietly *deletes* the sales rows for the missing products. An Associate that does not warn you has cost the client revenue. The [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief)'s failure <!-- plain-language: everyday Gallery -->gallery shows more.

```text
I have a sales table and a product-details table, and some products in the sales table are missing from the details table. Which `how=` should I use to add product details to sales, what happens to sales rows for the missing products, and how do I count how many I affected?
```

**A good answer** says `how="left"`. It warns that the default `how="inner"` silently drops the sales rows for the missing products. It counts the affected rows by checking for `NaN` in the detail columns after the merge, or by comparing row counts before and after.

## A first look at plotting

pandas plots a `DataFrame` through [Matplotlib](https://matplotlib.org/) with one method call. The [pandas visualisation guide](https://pandas.pydata.org/docs/user_guide/visualization.html) shows every chart type. These are quick diagnostic plots for you to get an idea; we'll cover this in much more detail in later sessions.

In [ ]:
import matplotlib.pyplot as plt

Create the figure and axes yourself with [`plt.subplots()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html). Pass the axes to pandas with `ax=`, so each chart gets its own figure. Otherwise pandas may draw onto whatever figure is already open.

In [ ]:
# Line plot: Monthly revenue over time (from the groupby above)
fig, ax = plt.subplots(figsize=(9, 4))
monthly_rev.plot(ax=ax, title="Monthly revenue, Jan-Jun 2026")
ax.set_ylabel("Revenue (£)")
plt.show()

A bar plot compares categories. Here it shows revenue by store, straight off a `groupby()` result.

In [ ]:
# Bar plot: Revenue by store
fig, ax = plt.subplots(figsize=(7, 4))
df_sales.groupby("store")["revenue"].sum().plot(
    kind="bar", ax=ax, title="Revenue by store")
ax.set_ylabel("Revenue (£)")
plt.show()

A histogram shows a column's *distribution*. Are daily revenues clustered around a typical day, or spread out?

In [ ]:
# Histogram: How are daily revenues distributed?
fig, ax = plt.subplots(figsize=(7, 4))
df_sales["revenue"].hist(bins=20, ax=ax)
ax.set_title("Distribution of daily revenue")
ax.set_xlabel("Revenue (£)")
plt.show()

A scatter plot puts two numeric columns against each other. Draw one first when you suspect a relationship.

In [ ]:
# Scatter plot: Do more units mean more revenue? (they should...)
fig, ax = plt.subplots(figsize=(7, 4))
df_sales.plot.scatter(x="units", y="revenue", ax=ax, title="Units vs revenue")
plt.show()

A box plot shows the spread of revenue *within* each category, outliers included, with five summaries per box.

In [ ]:
# Box plot: Revenue spread by category
fig, ax = plt.subplots(figsize=(7, 4))
df_sales.boxplot(column="revenue", by="category", ax=ax)
plt.suptitle("")
ax.set_title("Revenue by category")
ax.set_ylabel("Revenue (£)")
plt.show()

## Where we are ...

You can now create, inspect, filter, sort, combine, and summarise `DataFrames`, and read and write them in the three file formats you meet most.

You also have the checking habits that make AI-written pandas safe to sign off:

1. **`.dtypes` before you trust**: This week's main problem is text that should be numbers
2. **Count before and after**: Count the rows in and the rows out for every filter, duplicate drop, and merge
3. **`indicator=True` while developing merges**: This turns a silent loss into a visible one

Next, work through the [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/04/Quest.ipynb). Hartley & Voss have two ledgers that disagree, one merger, and a board meeting on Friday.